In [ ]:
import scanpy as sc, sys, os


run_dir = "/aloy/home/ddalton/projects/scGPT_playground/outputs/run-25-09-13-18"

adata_query = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-09-12-01/data.h5ad")
adata_valid = sc.read(os.path.join(run_dir, "adata_valid_1.h5ad"))

#! TEST-OUT
# filter genes we did not fine tune for!
# genes_ft = adata_query.var["gene_name"].tolist()
# mask_genes = np.isin(adata_query.var["gene_name"].tolist(), genes_ft )
# adata_query = adata_query[:, mask_genes]

# generate vocab
genes = adata_query.var["gene_name"].tolist()


In [10]:
def subset_adata(adata, adata_ref):
    import numpy as np
    import pandas as pd

    # mask presence in ref
    mask = np.isin(adata.obs.ids, adata_ref.obs.ids)
    adata_subset = adata[mask]

    # re-order to match
    desired = adata_ref.obs["ids"].tolist()
    cat = pd.Categorical(adata_subset.obs["ids"], categories=desired, ordered=True)
    adata_subset = adata_subset[adata_subset.obs.assign(_k=cat).sort_values("_k").index, :].copy()

    assert (adata_subset.obs["ids"].values == adata_ref.obs["ids"].values).all()

    return adata_subset


adata_subset = subset_adata(adata_query, adata_valid)


In [67]:
import sys
sys.path.append("../..")
from src.utils import utils as ut

# subset by disease
df_info = ut.load_dsa_info()

# map dsaid to disease id
dsa_to_disease_id = dict(zip(df_info["dsaid"], df_info["diseaseid"]))

# disease ids to keep
_dsaid = adata_valid.obs["dsaid"].to_list()
disease_ids = [dsa_to_disease_id[d] for d in _dsaid]
print(f"Nº of diseases : {len(set(disease_ids))}")

# get all dsaids w/ said diease ids
all_dsaids = [d for d, v in dsa_to_disease_id.items() if v in disease_ids]
print(f"Nº of dsaids : {len(set(_dsaid))}")

# filter adata
adata_query = adata_query[adata_query.obs["library"] == "Microarray"].copy()
adata_query = adata_query[adata_query.obs["dsaid"].isin(all_dsaids)].copy()
print("Using Microarray data only - adata shape", adata_query.shape)
print(f"Nº of diseases : {len(set(adata_query.obs['doid_id'].to_list()))}")

Nº of diseases : 114
Nº of dsaids : 547
Using Microarray data only - adata shape (61040, 20608)
Nº of diseases : 100


In [ ]:
import sys
sys.path.append("../..")
from src.utils import utils as ut

# subset by disease
df_info = ut.load_dsa_info()

# map dsaid to disease id
dsa_to_disease_id = dict(zip(df_info["dsaid"], df_info["diseaseid"]))

# disease ids to keep
_dsaid = adata_model.obs["dsaid"].to_list()
disease_ids = [dsa_to_disease_id[d] for d in _dsaid]
print(f"Nº of diseases : {len(set(disease_ids))}")

# get all dsaids w/ said diease ids
all_dsaids = [d for d, v in dsa_to_disease_id.items() if v in disease_ids]
print(f"Nº of dsaids : {len(set(_dsaid))}")

# filter adata
adata_query = adata_query[adata_query.obs["library"] == "Microarray"].copy()
adata_query = adata_query[adata_query.obs["dsaid"].isin(all_dsaids)].copy()
print("Using Microarray data only - adata shape", adata_query.shape)
print(f"Nº of diseases : {len(set(adata_query.obs['doid_id'].to_list()))}")

In [60]:
adata_valid.obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,celltype_id,sample_id,test_split_1
35169,DSA03144.GSM4114356.Control,GSE138614,GSE138614,506,148,DSA03144,Brain,19402,Control,Control,Multiple Sclerosis,RNA-Seq,D,Control,Control,Control,0,GSM4114356,0
16871,DSA01561.GSM3308428.Control,GSE117769,GSE117769,2057,567,DSA01561,Blood,19402,Control,Control,Rheumatoid Arthritis,RNA-Seq,D,Control,Control,Control,0,GSM3308428,0
44536,DSA03923.GSM4735789.Control,GSE156651,GSE156651,187,57,DSA03923,Whole blood,19402,Control,Control,Eosinophilic Esophagitis,RNA-Seq,D,Control,Control,Control,0,GSM4735789,0
116539,DSA09833.GSM4848882.Case,GSE159859,GSE159859,2116,585,DSA09833,nan,19402,Pitt-Hopkins Syndrome,DOID:0060488,Pitt-Hopkins Syndrome,RNA-Seq,D,DOID:0060488,DOID:0060488,Pitt-Hopkins syndrome,220,GSM4848882,0
73739,DSA06620.GSM2042128.Control,GSE76987,GSE76987,743,207,DSA06620,Colon,19402,Control,Control,Colon Cancer,RNA-Seq,D,Control,Control,Control,0,GSM2042128,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87329,DSA07497.GSM2373855.Case,GSE88805,GSE88805,1930,533,DSA07497,Stomach,19402,Gastric Adenocarcinoma,DOID:3717,Gastric Adenocarcinoma,RNA-Seq,D,DOID:3717,DOID:3717,gastric adenocarcinoma,78,GSM2373855,0
56961,DSA05072.GSM5978113.Case,GSE193677,GSE193677,1154,313,DSA05072,Bowel,19402,Ulcerative Colitis,DOID:8577,Ulcerative Colitis,RNA-Seq,D,DOID:8577,DOID:8577,ulcerative colitis,123,GSM5978113,0
55516,DSA05049.GSM5788632.Case,GSE193309,GSE193309,1802,495,DSA05049,Skin,19402,Atopic Dermatitis,DOID:3310,Atopic Dermatitis,RNA-Seq,D,DOID:3310,DOID:3310,atopic dermatitis,70,GSM5788632,0
62139,DSA05473.GSM6529135.Control,GSE212384,GSE212384,804,219,DSA05473,Blood,19402,Control,Control,Lung Disease,RNA-Seq,D,Control,Control,Control,0,GSM6529135,0


In [57]:
df_info

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,control_case_sample_count,definition
0,DSA00001,GSE224398,GPL21103,1000,Alzheimer's Disease,C0002395,Hippocampus,GEO,scRNA-Seq,Mus musculus,1|1,DO:An Alzheimer's disease that has_material_ba...
1,DSA00002,GSE224398,GPL21103,1000,Alzheimer's Disease,C0002395,Hippocampus,GEO,scRNA-Seq,Mus musculus,1|1,DO:An Alzheimer's disease that has_material_ba...
2,DSA00003,GSE224398,GPL21103,1000,Alzheimer's Disease,C0002395,Hippocampus,GEO,scRNA-Seq,Mus musculus,1|1,DO:An Alzheimer's disease that has_material_ba...
3,DSA00004,GSE224022,GPL16791,1000,Retinoblastoma,C0035335,Retina,GEO,RNA-Seq,Homo sapiens,4|5,DO:A retinal cell cancer and malignant neoplas...
4,DSA00005,GSE126342,GPL11154,1000,Congenital Myotonic Dystrophy,C0410226,Skeletal muscle,GEO,RNA-Seq,Homo sapiens,9|11,MONDO:An inherited progressive disorder affect...
...,...,...,...,...,...,...,...,...,...,...,...,...
10301,DSA10302,GSE6008,GPL96,1000,Ovarian Tumor,C1140680,Ovary,GEO,Microarray,Homo sapiens,4|41,DO:A female reproductive organ cancer that is ...
10302,DSA10303,GSE6280,GPL96,758,Kidney Tumor,C0022665,Kidney,GEO,Microarray,Homo sapiens,6|14,DO:A urinary system cancer that is located_in ...
10303,DSA10304,GSE6280,GPL97,337,Kidney Tumor,C0022665,Kidney,GEO,Microarray,Homo sapiens,6|14,DO:A urinary system cancer that is located_in ...
10304,DSA10305,GSE6344,GPL96,1000,Clear Cell Ependymoma,C1384403,Kidney,GEO,Microarray,Homo sapiens,5|5,"EFO:A WHO grade II, slow growing tumor of chil..."


In [13]:
adata_query

AnnData object with n_obs × n_vars = 121142 × 20608
    obs: 'ids', 'dataset', 'dataset_id', 'batch', 'batch_id', 'dsaid', 'tissue', 'n_genes', 'disease', 'celltype', 'disease_study', 'library', 'doid_study', 'doid_id', 'do_id', 'doid_disease'
    var: 'gene_symbols', 'gene_name', 'index'

In [19]:
adata_subset.X

array([[1.180e+02, 4.000e+00, 1.505e+03, ..., 5.300e+01, 3.542e+03,
        2.944e+03],
       [6.500e+01, 0.000e+00, 1.000e+00, ..., 1.000e+01, 1.133e+03,
        2.290e+02],
       [1.000e+01, 0.000e+00, 7.600e+01, ..., 7.000e+00, 2.030e+02,
        1.350e+02],
       ...,
       [7.800e+01, 0.000e+00, 3.934e+03, ..., 8.200e+01, 8.010e+02,
        4.830e+02],
       [8.000e+01, 4.000e+00, 3.444e+03, ..., 1.810e+02, 2.574e+03,
        3.610e+02],
       [1.600e+01, 1.000e+00, 4.050e+03, ..., 1.000e+00, 8.870e+02,
        5.200e+02]])

In [18]:
adata_valid.X

array([[5.330e+02, 4.760e+02, 3.080e+02, ..., 1.232e+03, 1.630e+03,
        5.300e+01],
       [1.490e+02, 7.500e+01, 0.000e+00, ..., 3.800e+02, 4.320e+02,
        1.000e+01],
       [1.900e+02, 2.500e+01, 0.000e+00, ..., 2.970e+02, 1.370e+02,
        7.000e+00],
       ...,
       [1.169e+03, 1.619e+03, 3.300e+01, ..., 7.601e+03, 5.540e+02,
        8.200e+01],
       [2.140e+02, 1.050e+02, 5.000e+00, ..., 4.980e+02, 3.830e+02,
        1.810e+02],
       [4.790e+02, 8.600e+02, 3.900e+01, ..., 6.000e+02, 2.140e+02,
        1.000e+00]])

In [ ]:
import numpy as np
mask_genes = np.isin(adata_query.var["gene_name"].tolist(), adata_valid.var["gene_name"].tolist() )

_adata_query = adata_subset[:, mask_genes]

(_adata_query.X == adata_valid.X).sum(axis=0)

array([2034, 2034, 2034, ..., 2034, 2034, 2034])

In [33]:
np.argwhere((_adata_query.X == adata_valid.X).sum(axis=1)!=3501)

array([[ 815],
       [1268],
       [1317],
       [1392],
       [1509],
       [1528]])

In [39]:
np.where(~(adata_valid.X[815] == _adata_query.X[815]))

(array([ 151,  152,  153,  760,  949, 1654]),)

In [47]:
type(_adata_query.X[815][949])

numpy.float64

In [48]:
type(adata_valid.X[815][949])

numpy.float64

In [27]:
_adata_query.X.shape[0]*_adata_query.X.shape[1]

7121034

In [7]:
import numpy as np
mask_genes = np.isin(adata_query.var["gene_name"].tolist(), adata_valid.var["gene_name"].tolist() )

_adata_query = adata_query[:, mask_genes]

In [9]:
_adata_query.X

ArrayView([[446.        ,  94.        ,  68.        , ..., 253.        ,
            337.        ,  20.        ],
           [242.        , 125.        ,  61.        , ..., 227.        ,
            338.        ,   8.        ],
           [289.        , 113.        , 123.        , ..., 354.        ,
            312.        ,   4.        ],
           ...,
           [  5.36127908,   6.22240836,          nan, ...,  11.93176167,
                     nan,   9.02554381],
           [  5.63071658,   6.48870129,          nan, ...,  13.11040667,
                     nan,  10.34025538],
           [  6.51386934,   5.75105169,          nan, ...,  12.83533013,
                     nan,   8.32404382]])

In [1]:
ls /aloy/home/ddalton/projects/scGPT_playground/outputs/run-25-09-13-18

adata_orig.h5ad                args.json           predictions_test.pkl
adata_orig.multilabels.pkl     d_metrics.pkl       predictions_train.pkl
adata_test_1.h5ad              id2type.pkl         predictions_valid.pkl
adata_test_1.multilabels.pkl   labels_test.pkl     results_test.pkl
adata_train_1.h5ad             labels_train.pkl    results_train.pkl
adata_train_1.multilabels.pkl  labels_valid.pkl    results_valid.pkl
adata_valid_1.h5ad             metrics_epochs.pkl  split.pkl
adata_valid_1.multilabels.pkl  model_1.pt          train_indices.pkl
all_outputs_test.pkl           old_args.json       valid_indices.pkl
all_outputs_train.pkl          outputs/            vocab.json
all_outputs_valid.pkl          parameters.json
